In [9]:
%run tree.py
%run cvar_tree_utilities.py
%run regression_tree.py

In [10]:
N_list = [100, 200, 400]; p = 10
R = 0.1; alpha = 0.1; obj_coef = 0
lb = 0; ub = 1;  sum_bound = 1; if_stoch_constr = False
runs = 10

generate_Y = generate_Y_lognormal
cond_mean = [lambda x: np.exp(x[:, 0])/5, lambda x: x[:, 0]/5, lambda x: np.abs(x[:, 0])/5]
cond_std = [lambda x: 1 - 0.5*((-3<=x[:, 1]) & (x[:, 1]<=-1)), lambda x: 1 - 0.5*((-1<=x[:, 1])&(x[:, 1]<=1)), lambda x: 1 - 0.5*((1<=x[:, 1])&(x[:, 1]<=3))]

opt_solver = partial(solve_cvar, alpha = alpha, R = R, obj_coef = obj_coef, lb = lb, ub = ub, sum_bound = sum_bound, if_stoch_constr = if_stoch_constr)
hessian_computer = partial(compute_hessian, alpha = alpha)
active_constraint = partial(search_active_constraint,  R = R, lb = lb, ub = ub, sum_bound = sum_bound, if_stoch_constr = if_stoch_constr)
gradient_computer = partial(compute_gradient,  alpha = alpha, R = R, obj_coef = obj_coef)
update_step = partial(compute_update_step, R = R)

time_list = {str(N): {key: np.zeros(runs) for key in ["rf_approx_risk", "rf_approx_sol", "rf_rf", "rf_oracle", "rf_rf_true"]} for N in N_list}

In [11]:
for N in N_list:
    print("N:", N)
    n_proposals = N; 
    mtry = p;
    subsample_ratio = 1;
    max_depth=100; 
    min_leaf_size=10; 
    balancedness_tol = 0.2; 
    honesty = False;
    verbose = False; oracle = True;
    bootstrap = True; 
    
    X_list = [np.random.normal(size = (N, p)) for run in range(runs)]
    Y_list = [generate_Y(X_list[run], cond_mean, cond_std) for run in range(runs)]

    for run in range(runs):
        print("run:", run)
        Y = Y_list[run]; Y_est = Y_list[run]
        X = X_list[run]; X_est = X_list[run]; 
        
        time1 = time.time()
        rf_approx_risk = build_tree(Y, X, Y_est, X_est, 
                             opt_solver = opt_solver, hessian_computer = hessian_computer,
                             gradient_computer = gradient_computer, 
                             search_active_constraint = active_constraint,
                             compute_update_step = update_step,
                             crit_computer = compute_crit_approx_risk, 
                             honesty = honesty, mtry = mtry,
                             min_leaf_size = min_leaf_size, max_depth = max_depth, 
                             n_proposals = n_proposals, balancedness_tol = balancedness_tol,
                             verbose = verbose)
        time2 = time.time()
        time_list[str(N)]["rf_approx_risk"][run] = time2 - time1

        time1 = time.time()
        rf_approx_sol = build_tree(Y, X, Y_est, X_est, 
                             opt_solver = opt_solver, hessian_computer = hessian_computer,
                             gradient_computer = gradient_computer, 
                             search_active_constraint = active_constraint,
                             compute_update_step = update_step,
                             crit_computer = partial(compute_crit_approx_sol, obj_coef = obj_coef, alpha = alpha), 
                             honesty = honesty, mtry = mtry,
                             min_leaf_size = min_leaf_size, max_depth = max_depth, 
                             n_proposals = n_proposals, balancedness_tol = balancedness_tol,
                             verbose = verbose)
        time2 = time.time()
        time_list[str(N)]["rf_approx_sol"][run] = time2 - time1

        time1 = time.time()
        rf_rf = build_tree(Y, X, Y_est, X_est, 
                             opt_solver = opt_solver, hessian_computer = hessian_computer,
                             gradient_computer = gradient_computer, 
                             search_active_constraint = active_constraint,
                             compute_update_step = update_step,
                             crit_computer = compute_crit_rf, 
                             honesty = honesty, mtry = mtry,
                             min_leaf_size = min_leaf_size, max_depth = max_depth, 
                             n_proposals = n_proposals, balancedness_tol = balancedness_tol,
                             verbose = verbose)
        time2 = time.time()
        time_list[str(N)]["rf_rf"][run] = time2 - time1
    
        time1 = time.time()
        rf_oracle = build_tree(Y, X, Y_est, X_est, 
                             opt_solver = opt_solver, hessian_computer = hessian_computer,
                             gradient_computer = gradient_computer, 
                             search_active_constraint = active_constraint,
                             compute_update_step = update_step,
                             crit_computer = partial(compute_crit_oracle, solver = opt_solver), 
                             honesty = honesty, mtry = mtry,
                             min_leaf_size = min_leaf_size, max_depth = max_depth, 
                             n_proposals = n_proposals, balancedness_tol = balancedness_tol,
                             verbose = verbose)
        time2 = time.time()
        time_list[str(N)]["rf_oracle"][run] = time2 - time1
        
        time1 = time.time()
        rf_oracle = build_reg_tree(Y, X, Y_est, X_est,
                             honesty = honesty, mtry = mtry,
                             min_leaf_size = min_leaf_size, max_depth = max_depth, 
                             n_proposals = n_proposals, balancedness_tol = balancedness_tol,
                             verbose = verbose)
        time2 = time.time()
        time_list[str(N)]["rf_rf_true"][run] = time2 - time1
    

N: 100
run: 0


GurobiError: License expired 2020-09-26 - license file '/Users/xiaojicmao/gurobi.lic'

In [19]:
pickle.dump(time_list, open("time_cvar.pkl", "wb"))

In [27]:
{N: {m: time_list[N][m].mean() for m in time_list[N].keys()} for N in time_list.keys()}

{'100': {'rf_approx_risk': 0.25630850791931153,
  'rf_approx_sol': 0.21890861988067628,
  'rf_oracle': 41.41384615898132,
  'rf_rf': 0.21552760601043702},
 '200': {'rf_approx_risk': 0.680793023109436,
  'rf_approx_sol': 0.6992438793182373,
  'rf_oracle': 165.025603890419,
  'rf_rf': 0.49462893009185793},
 '400': {'rf_approx_risk': 1.6779053449630736,
  'rf_approx_sol': 2.238886022567749,
  'rf_oracle': 615.8862199783325,
  'rf_rf': 1.3391101837158204}}

In [28]:
{N: {m: time_list[N][m].std() for m in time_list[N].keys()} for N in time_list.keys()}

{'100': {'rf_approx_risk': 0.07662490427581575,
  'rf_approx_sol': 0.05256373332214826,
  'rf_oracle': 5.432989145381002,
  'rf_rf': 0.08287181305335586},
 '200': {'rf_approx_risk': 0.36117759754177,
  'rf_approx_sol': 0.2021090993018887,
  'rf_oracle': 15.774651640318456,
  'rf_rf': 0.06514774768712296},
 '400': {'rf_approx_risk': 0.5416776652506486,
  'rf_approx_sol': 0.33419189845281644,
  'rf_oracle': 83.91081487211585,
  'rf_rf': 0.45480064096023876}}

In [12]:
for N in N_list:
    print("N:", N)
    n_proposals = N; 
    mtry = p;
    subsample_ratio = 1;
    max_depth=100; 
    min_leaf_size=10; 
    balancedness_tol = 0.2; 
    honesty = False;
    verbose = False; oracle = True;
    bootstrap = True; 
    
    X_list = [np.random.normal(size = (N, p)) for run in range(runs)]
    Y_list = [generate_Y(X_list[run], cond_mean, cond_std) for run in range(runs)]

    for run in range(runs):
        print("run:", run)
        Y = Y_list[run]; Y_est = Y_list[run]
        X = X_list[run]; X_est = X_list[run]; 

        
        time1 = time.time()
        rf_oracle = build_reg_tree(Y, X, Y_est, X_est,
                             honesty = honesty, mtry = mtry,
                             min_leaf_size = min_leaf_size, max_depth = max_depth, 
                             n_proposals = n_proposals, balancedness_tol = balancedness_tol,
                             verbose = verbose)
        time2 = time.time()
        time_list[str(N)]["rf_rf_true"][run] = time2 - time1
    

N: 100
run: 0
run: 1
run: 2
run: 3
run: 4
run: 5
run: 6
run: 7
run: 8
run: 9
N: 200
run: 0
run: 1
run: 2
run: 3
run: 4
run: 5
run: 6
run: 7
run: 8
run: 9
N: 400
run: 0
run: 1
run: 2
run: 3
run: 4
run: 5
run: 6
run: 7
run: 8
run: 9


In [14]:
{N: {m: time_list[N][m].mean() for m in time_list[N].keys()} for N in time_list.keys()}

{'100': {'rf_approx_risk': 0.0,
  'rf_approx_sol': 0.0,
  'rf_rf': 0.0,
  'rf_oracle': 0.0,
  'rf_rf_true': 0.01376514434814453},
 '200': {'rf_approx_risk': 0.0,
  'rf_approx_sol': 0.0,
  'rf_rf': 0.0,
  'rf_oracle': 0.0,
  'rf_rf_true': 0.04281761646270752},
 '400': {'rf_approx_risk': 0.0,
  'rf_approx_sol': 0.0,
  'rf_rf': 0.0,
  'rf_oracle': 0.0,
  'rf_rf_true': 0.26560900211334226}}

In [13]:
{N: {m: time_list[N][m].std() for m in time_list[N].keys()} for N in time_list.keys()}

{'100': {'rf_approx_risk': 0.0,
  'rf_approx_sol': 0.0,
  'rf_rf': 0.0,
  'rf_oracle': 0.0,
  'rf_rf_true': 0.0027231130156801526},
 '200': {'rf_approx_risk': 0.0,
  'rf_approx_sol': 0.0,
  'rf_rf': 0.0,
  'rf_oracle': 0.0,
  'rf_rf_true': 0.0065925301424771465},
 '400': {'rf_approx_risk': 0.0,
  'rf_approx_sol': 0.0,
  'rf_rf': 0.0,
  'rf_oracle': 0.0,
  'rf_rf_true': 0.05117350823009773}}